# Model Training

This notebook is optimized for the final deadline sprint.

It does four things:

1. builds a strict grouped evaluation pipeline using the existing `Region` column,
2. tests a small shortlist of target-specific candidates,
3. freezes a safe manifest from full grouped CV,
4. writes three submission files:
   - **A** = safe anchor,
   - **B** = EC aggressive + DRP safe,
   - **C** = hedge blend.

Notes:
- We assume `Region` already exists in the provided dataset.
- We keep the notebook cell-by-cell and avoid one giant integrated script.
- We clip predictions to nonnegative values before submission.

### Lean Run Guide
Run all cells top-to-bottom, but optional diagnostics are pre-commented in cells `9`, `25`, `28`, and `34` for faster execution.


In [15]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from IPython.display import display
from tqdm.auto import tqdm


## Environment and MLflow

In [16]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', config.MLFLOW_URI)

MLflow URI: sqlite:///../mlflow.db


## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [17]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'SpatialGroupKFold+PseudoHoldoutGroups'
GROUP_DEFINITION_VERSION = 'kmeans_latlon_v2_group_holdout'
PIPELINE_VERSION = 'deadline_v2_mvp4_spatial_group_holdout'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline_mvp4'

SPATIAL_N_CLUSTERS = 16
CV_N_SPLITS = 5
HOLDOUT_MARGIN_DEG = 0.35
HOLDOUT_MIN_GROUPS = 3
HOLDOUT_MIN_FRAC = 0.08
HOLDOUT_MAX_FRAC = 0.15

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

Global config loaded.


## Data loading

We load the training data and confirm that the `Region` column already exists.

In [18]:
TRAIN_PATH_CANDIDATES = [
    '../data/interim/master_train_osm_aligned.parquet',
    '../data/interim/master_train_osm.parquet',
    '../data/interim/water_quality_mvp_baseline.parquet',
]
VALID_PATH_CANDIDATES = [
    '../data/interim/master_test_osm_aligned.parquet',
    '../data/interim/master_test_osm.parquet',
    '../data/interim/water_quality_mvp_validation.parquet',
]
CONTRACT_TXT_PATH = '../data/interim/feature_contract_master_osm.txt'


def pick_first_existing(paths, label):
    for path in paths:
        if os.path.exists(path):
            print(f'{label} selected: {path}')
            return path
    raise RuntimeError(f'No existing path found for {label}: {paths}')


TRAIN_PATH = pick_first_existing(TRAIN_PATH_CANDIDATES, 'TRAIN_PATH')
VALID_PATH = pick_first_existing(VALID_PATH_CANDIDATES, 'VALID_PATH')


df = pd.read_parquet(TRAIN_PATH).copy()
df_val_all = pd.read_parquet(VALID_PATH).copy()

required_cols = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
missing_cols_train = [c for c in required_cols if c not in df.columns]
if missing_cols_train:
    raise RuntimeError(f'Missing required training columns: {missing_cols_train}')

missing_cols_valid_geo = [c for c in ['Latitude', 'Longitude', 'Sample Date'] if c not in df_val_all.columns]
if missing_cols_valid_geo:
    raise RuntimeError(f'Missing required validation geo/time columns: {missing_cols_valid_geo}')

# Optional schema-contract guard (produced by 01_eda_and_discovery / 02_preprocessing)
if os.path.exists(CONTRACT_TXT_PATH):
    with open(CONTRACT_TXT_PATH, 'r', encoding='utf-8') as f:
        contract_cols = [line.strip() for line in f.readlines() if line.strip()]

    missing_contract_train = [c for c in contract_cols if c not in df.columns]
    missing_contract_valid = [c for c in contract_cols if c not in df_val_all.columns]

    if missing_contract_train:
        raise RuntimeError(f'Contract columns missing in train ({len(missing_contract_train)}): {missing_contract_train[:20]}')
    if missing_contract_valid:
        raise RuntimeError(f'Contract columns missing in validation ({len(missing_contract_valid)}): {missing_contract_valid[:20]}')

    print(f'Contract check passed: {len(contract_cols)} feature columns present in train/validation.')
else:
    print('Contract TXT not found; continuing without contract schema guard.')


df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce')
df_val_geo = df_val_all[['Latitude', 'Longitude', 'Sample Date']].copy()
df_val_geo['Sample Date'] = pd.to_datetime(df_val_geo['Sample Date'], errors='coerce')


def add_spatial_groups(data: pd.DataFrame, n_clusters: int = SPATIAL_N_CLUSTERS) -> pd.DataFrame:
    '''
    Create stable spatial groups from latitude/longitude using KMeans.
    '''
    out = data.copy()
    n_clusters = min(max(4, int(n_clusters)), len(out))

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    out['spatial_group'] = km.fit_predict(out[['Latitude', 'Longitude']].astype(float)).astype(str)
    return out


def select_pseudo_holdout_groups(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    min_groups: int = HOLDOUT_MIN_GROUPS,
    min_frac: float = HOLDOUT_MIN_FRAC,
    max_frac: float = HOLDOUT_MAX_FRAC,
    margin_deg: float = HOLDOUT_MARGIN_DEG,
):
    '''
    Select whole spatial groups nearest to the validation footprint.
    The selection targets a row fraction range and enforces a minimum number of groups.
    '''
    gdf = train_df.groupby('spatial_group', as_index=False).agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        n=('spatial_group', 'size')
    )

    lat_min = float(valid_df['Latitude'].min()) - margin_deg
    lat_max = float(valid_df['Latitude'].max()) + margin_deg
    lon_min = float(valid_df['Longitude'].min()) - margin_deg
    lon_max = float(valid_df['Longitude'].max()) + margin_deg

    valid_center_lat = float(valid_df['Latitude'].mean())
    valid_center_lon = float(valid_df['Longitude'].mean())

    lat = gdf['Latitude'].astype(float)
    lon = gdf['Longitude'].astype(float)

    lat_gap = np.maximum(np.maximum(lat_min - lat, 0.0), lat - lat_max)
    lon_gap = np.maximum(np.maximum(lon_min - lon, 0.0), lon - lon_max)

    gdf['bbox_dist'] = np.sqrt(lat_gap ** 2 + lon_gap ** 2)
    gdf['center_dist'] = np.sqrt((lat - valid_center_lat) ** 2 + (lon - valid_center_lon) ** 2)

    gdf = gdf.sort_values(['bbox_dist', 'center_dist', 'n'], ascending=[True, True, False]).reset_index(drop=True)

    total_rows = int(len(train_df))
    selected = []
    selected_rows = 0

    for _, row in gdf.iterrows():
        group_name = str(row['spatial_group'])
        group_rows = int(row['n'])

        need_groups = len(selected) < int(min_groups)
        need_rows = (selected_rows / total_rows) < float(min_frac)

        if need_groups or need_rows:
            selected.append(group_name)
            selected_rows += group_rows
            continue

        next_frac = (selected_rows + group_rows) / total_rows
        if next_frac <= float(max_frac):
            selected.append(group_name)
            selected_rows += group_rows
        else:
            break

    i = len(selected)
    while (len(selected) < int(min_groups) or (selected_rows / total_rows) < float(min_frac)) and i < len(gdf):
        group_name = str(gdf.loc[i, 'spatial_group'])
        if group_name not in selected:
            selected.append(group_name)
            selected_rows += int(gdf.loc[i, 'n'])
        i += 1

    return selected


# Build spatial groups and pseudo-holdout mask
# This must run before grouped_oof_eval (which expects `spatial_group`).
df = add_spatial_groups(df, n_clusters=SPATIAL_N_CLUSTERS)

holdout_groups = select_pseudo_holdout_groups(
    train_df=df,
    valid_df=df_val_geo,
    min_groups=HOLDOUT_MIN_GROUPS,
    min_frac=HOLDOUT_MIN_FRAC,
    max_frac=HOLDOUT_MAX_FRAC,
    margin_deg=HOLDOUT_MARGIN_DEG,
)

if not holdout_groups:
    raise RuntimeError('Pseudo-holdout selection returned no groups.')

holdout_group_set = set(pd.Series(holdout_groups).astype(str).tolist())
df['is_pseudo_valid'] = df['spatial_group'].astype(str).isin(holdout_group_set)

print('Spatial grouping ready.')
print('n_rows:', len(df))
print('n_spatial_groups:', int(df['spatial_group'].nunique()))
print('holdout_groups:', sorted(list(holdout_group_set)))
print('holdout_rows:', int(df['is_pseudo_valid'].sum()), '| holdout_frac:', round(float(df['is_pseudo_valid'].mean()), 4))

# Guard against group leakage between pseudo-holdout and train subsets
train_groups = set(df.loc[~df['is_pseudo_valid'], 'spatial_group'].astype(str).tolist())
test_groups = set(df.loc[df['is_pseudo_valid'], 'spatial_group'].astype(str).tolist())
if train_groups.intersection(test_groups):
    raise RuntimeError('Pseudo-holdout leakage detected after split setup.')


TRAIN_PATH selected: ../data/interim/master_train_osm_aligned.parquet
VALID_PATH selected: ../data/interim/master_test_osm_aligned.parquet
Contract check passed: 188 feature columns present in train/validation.
Spatial grouping ready.
n_rows: 9319
n_spatial_groups: 16
holdout_groups: ['10', '3', '9']
holdout_rows: 1433 | holdout_frac: 0.1538


In [26]:
# OPTIONAL DIAGNOSTIC (disabled for faster runs)
# print(df.info())
# Uncomment above if you need full schema summary before training.


## Feature engineering

This notebook only uses the two engineered features that were explicitly confirmed from the winning setup:

- `pop_density_upstream`
- `specific_discharge`

In [19]:
# Feature-engineering ablation toggles
FE_INCLUDE_CLASS_DELTAS = False  # quick ablation: disable noisy per-class SANLC deltas


def _safe_div(a: pd.Series, b: pd.Series, eps: float = 1e-6) -> pd.Series:
    return a.astype(float) / (b.astype(float) + eps)


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    '''
    High-ROI engineered features from current schema:
    - temporal cyclics (if date/month exists)
    - hydroclimate interactions (Terra + weather)
    - soil/hydro/population interactions
    - OSM pressure/proximity transforms + interactions
    - SANLC thematic aggregate shares + deltas
    Optional:
    - per-class SANLC deltas (controlled by FE_INCLUDE_CLASS_DELTAS)
    '''
    out = data.copy()

    # Clean previously engineered columns if function is re-run in same kernel state.
    exact_drop = {
        'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'is_wet_season',
        'aridity_idx', 'water_balance', 'evap_eff', 'runoff_ratio', 'dry_heat', 'temp_range',
        'rain_wind_event', 'soil_texture_balance', 'soil_fines', 'drainage_proxy',
        'upstream_human_pressure', 'stream_power_proxy', 'river_discharge_per_width',
        'hydro_pressure_index', 'pop_compaction_1km', 'pop_gradient', 'pop_sum_ratio_1_to_5km',
        'human_land_share_2020', 'human_land_share_2022', 'human_land_share_delta',
        'osm_total_pressure', 'farm_rain', 'farm_runoff', 'mine_runoff', 'ww_dry',
        'ww_urban', 'farm_crop', 'mine_land',
    }
    drop_cols = [
        c for c in out.columns
        if c in exact_drop
        or c.startswith('sanlc_delta_')
        or c.startswith('sanlc_abs_delta_')
        or c.startswith('osm_log_count_')
        or c.startswith('osm_log_dist_')
        or c.startswith('osm_inv_dist_')
        or c.startswith('osm_local_ratio_')
        or c.startswith('osm_ring_count_')
        or c.startswith('osm_pressure_')
        or c.startswith('osm_near_')
        or (c.startswith('sanlc_') and ('_share_2020' in c or '_share_2022' in c or c.endswith('_delta') or '_ratio_' in c))
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols, errors='ignore')

    # -------------------------
    # 1) Temporal cyclic features
    # -------------------------
    month = None
    doy = None

    date_col = None
    for cand in ['Sample_Date', 'Sample Date', 'Date', 'date']:
        if cand in out.columns:
            date_col = cand
            break

    if date_col is not None:
        dt = pd.to_datetime(out[date_col], errors='coerce')
        month = dt.dt.month.astype(float)
        doy = dt.dt.dayofyear.astype(float)
    elif 'Month' in out.columns:
        month = pd.to_numeric(out['Month'], errors='coerce').astype(float)
    elif 'month' in out.columns:
        month = pd.to_numeric(out['month'], errors='coerce').astype(float)

    if month is not None:
        out['month_sin'] = np.sin(2.0 * np.pi * (month / 12.0))
        out['month_cos'] = np.cos(2.0 * np.pi * (month / 12.0))
        wet_months = {11, 12, 1, 2, 3}
        out['is_wet_season'] = np.where(month.isna(), np.nan, month.isin(wet_months).astype(float))

    if doy is not None:
        out['doy_sin'] = np.sin(2.0 * np.pi * (doy / 366.0))
        out['doy_cos'] = np.cos(2.0 * np.pi * (doy / 366.0))

    # -------------------------
    # 2) Hydroclimate interactions (Terra + weather)
    # -------------------------
    if {'terra_pet', 'terra_ppt'}.issubset(out.columns):
        pet = out['terra_pet'].fillna(0.0)
        ppt = out['terra_ppt'].fillna(0.0)
        out['aridity_idx'] = _safe_div(pet, ppt)
        out['water_balance'] = ppt - pet

    if {'terra_aet', 'terra_pet'}.issubset(out.columns):
        out['evap_eff'] = _safe_div(out['terra_aet'].fillna(0.0), out['terra_pet'].fillna(0.0))

    if {'terra_q', 'terra_ppt'}.issubset(out.columns):
        out['runoff_ratio'] = _safe_div(out['terra_q'].fillna(0.0), out['terra_ppt'].fillna(0.0))

    if {'terra_vpd', 'terra_tmax'}.issubset(out.columns):
        out['dry_heat'] = out['terra_vpd'].fillna(0.0).astype(float) * out['terra_tmax'].fillna(0.0).astype(float)

    if {'terra_tmax', 'terra_tmin'}.issubset(out.columns):
        out['temp_range'] = out['terra_tmax'].fillna(0.0).astype(float) - out['terra_tmin'].fillna(0.0).astype(float)

    if {'weather_precip_7d_sum', 'weather_wind_7d_mean'}.issubset(out.columns):
        out['rain_wind_event'] = (
            out['weather_precip_7d_sum'].fillna(0.0).astype(float)
            * out['weather_wind_7d_mean'].fillna(0.0).astype(float)
        )

    # -------------------------
    # 3) Soil / hydro / population interactions
    # -------------------------
    if {'soil_sand_mean_0_5cm', 'soil_clay_mean_0_5cm'}.issubset(out.columns):
        out['soil_texture_balance'] = (
            out['soil_sand_mean_0_5cm'].fillna(0.0).astype(float)
            - out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        )

    if {'soil_clay_mean_0_5cm', 'soil_silt_mean_0_5cm'}.issubset(out.columns):
        clay = out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        silt = out['soil_silt_mean_0_5cm'].fillna(0.0).astype(float)
        out['soil_fines'] = clay + silt

    if {'soil_sand_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_silt_mean_0_5cm'}.issubset(out.columns):
        sand = out['soil_sand_mean_0_5cm'].fillna(0.0).astype(float)
        clay = out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        silt = out['soil_silt_mean_0_5cm'].fillna(0.0).astype(float)
        out['drainage_proxy'] = _safe_div(sand, clay + silt)

    if {'basin_population', 'basin_upstream_area_km2'}.issubset(out.columns):
        pop = out['basin_population'].fillna(0.0).astype(float).clip(lower=0.0)
        area = out['basin_upstream_area_km2'].fillna(0.0).astype(float).clip(lower=0.0)
        out['upstream_human_pressure'] = np.log1p(pop) / (np.log1p(area) + 1e-6)

    if {'dem_slope_1km', 'river_avg_discharge_cms'}.issubset(out.columns):
        slope = out['dem_slope_1km'].fillna(0.0).astype(float)
        q = out['river_avg_discharge_cms'].fillna(0.0).astype(float)
        out['stream_power_proxy'] = slope * q

    if {'river_avg_discharge_cms', 'river_width_m'}.issubset(out.columns):
        out['river_discharge_per_width'] = _safe_div(
            out['river_avg_discharge_cms'].fillna(0.0).astype(float),
            out['river_width_m'].fillna(0.0).astype(float),
        )

    if {'stream_power_proxy', 'upstream_human_pressure'}.issubset(out.columns):
        out['hydro_pressure_index'] = (
            out['stream_power_proxy'].fillna(0.0).astype(float)
            * out['upstream_human_pressure'].fillna(0.0).astype(float)
        )

    if {'worldpop_max_1km', 'worldpop_mean_1km'}.issubset(out.columns):
        out['pop_compaction_1km'] = _safe_div(
            out['worldpop_max_1km'].fillna(0.0).astype(float),
            out['worldpop_mean_1km'].fillna(0.0).astype(float),
        )

    if {'worldpop_mean_1km', 'worldpop_mean_5km'}.issubset(out.columns):
        out['pop_gradient'] = (
            out['worldpop_mean_1km'].fillna(0.0).astype(float)
            - out['worldpop_mean_5km'].fillna(0.0).astype(float)
        )

    if {'worldpop_sum_1km', 'worldpop_sum_5km'}.issubset(out.columns):
        out['pop_sum_ratio_1_to_5km'] = _safe_div(
            out['worldpop_sum_1km'].fillna(0.0).astype(float),
            out['worldpop_sum_5km'].fillna(0.0).astype(float),
        )

    # -------------------------
    # 4) OSM transforms and source pressure
    # -------------------------
    osm_map = {
        'mine': ('osm_dist_nearest_mine', 'osm_mine_count_1km', 'osm_mine_count_5km'),
        'farm': ('osm_dist_nearest_farm', 'osm_farm_count_1km', 'osm_farm_count_5km'),
        'wastewater': ('osm_dist_nearest_wastewater', 'osm_wastewater_count_1km', 'osm_wastewater_count_5km'),
    }

    for src, (dist_col, c1_col, c5_col) in osm_map.items():
        has_dist = dist_col in out.columns
        has_c1 = c1_col in out.columns
        has_c5 = c5_col in out.columns

        if has_dist:
            dist = out[dist_col].fillna(0.0).astype(float).clip(lower=0.0)
            out[f'osm_log_dist_{src}'] = np.log1p(dist)
            out[f'osm_inv_dist_{src}'] = 1.0 / (1.0 + dist)
            out[f'osm_near_{src}_lt1km'] = (dist <= 1000.0).astype(float)
            out[f'osm_near_{src}_lt5km'] = (dist <= 5000.0).astype(float)

        if has_c1:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            out[f'osm_log_count_{src}_1km'] = np.log1p(c1)

        if has_c5:
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            out[f'osm_log_count_{src}_5km'] = np.log1p(c5)

        if has_c1 and has_c5:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            out[f'osm_local_ratio_{src}'] = _safe_div(c1, c5 + 1.0)
            out[f'osm_ring_count_{src}'] = (c5 - c1).clip(lower=0.0)

        has_pressure_inputs = has_dist and has_c1 and has_c5
        if has_pressure_inputs:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            inv_d = out[f'osm_inv_dist_{src}']
            out[f'osm_pressure_{src}'] = np.log1p(c1) + 0.5 * np.log1p(c5) + inv_d

    pressure_cols = [f'osm_pressure_{s}' for s in ['mine', 'farm', 'wastewater'] if f'osm_pressure_{s}' in out.columns]
    if pressure_cols:
        out['osm_total_pressure'] = out[pressure_cols].sum(axis=1)

    if {'osm_pressure_farm', 'weather_precip_7d_sum'}.issubset(out.columns):
        out['farm_rain'] = out['osm_pressure_farm'].fillna(0.0) * out['weather_precip_7d_sum'].fillna(0.0)

    if {'osm_pressure_farm', 'terra_q'}.issubset(out.columns):
        out['farm_runoff'] = out['osm_pressure_farm'].fillna(0.0) * out['terra_q'].fillna(0.0)

    if {'osm_pressure_mine', 'terra_q'}.issubset(out.columns):
        out['mine_runoff'] = out['osm_pressure_mine'].fillna(0.0) * out['terra_q'].fillna(0.0)

    if 'osm_pressure_wastewater' in out.columns:
        if 'aridity_idx' in out.columns:
            dry_ref = out['aridity_idx'].fillna(0.0)
        elif 'terra_vpd' in out.columns:
            dry_ref = out['terra_vpd'].fillna(0.0)
        else:
            dry_ref = None
        if dry_ref is not None:
            out['ww_dry'] = out['osm_pressure_wastewater'].fillna(0.0) * dry_ref

    # -------------------------
    # 5) SANLC aggregate themes + deltas
    # -------------------------
    p20 = 'sanlc2020_pct_'
    p22 = 'sanlc2022_pct_'

    cols20 = [c for c in out.columns if c.startswith(p20)]
    cols22 = [c for c in out.columns if c.startswith(p22)]

    suffix20 = {c[len(p20):] for c in cols20}
    suffix22 = {c[len(p22):] for c in cols22}
    common_suffixes = sorted(suffix20.intersection(suffix22))

    if FE_INCLUDE_CLASS_DELTAS:
        for suf in common_suffixes:
            c20 = p20 + suf
            c22 = p22 + suf
            v20 = out[c20].fillna(0.0).astype(float)
            v22 = out[c22].fillna(0.0).astype(float)
            delta = v22 - v20
            out[f'sanlc_delta_{suf}'] = delta
            out[f'sanlc_abs_delta_{suf}'] = delta.abs()

    theme_keywords = {
        'urban': ['urban', 'residential', 'village', 'settlement', 'roads', 'rails', 'industrial', 'built'],
        'mining': ['mine', 'mines', 'tailings', 'resource_dumps', 'quarr', 'extraction_pits'],
        'cropland': ['crop', 'crops', 'cultivated', 'orchard', 'vineyard', 'fallow', 'smallholding', 'sugarcane'],
        'wetland': ['wetland', 'wetlands', 'marsh', 'peat'],
        'water': ['river', 'rivers', 'dam', 'dams', 'canal', 'water', 'estuar', 'lagoon', 'pans'],
        'bare': ['bare', 'rock', 'riverbed', 'sand'],
        'forest': ['forest', 'woodland', 'thicket'],
        'grass': ['grassland', 'grass', 'herbaceous', 'shrubland', 'bush'],
    }

    def cols_for_theme(year_cols, keywords):
        selected = []
        for c in year_cols:
            cl = c.lower()
            if any(k in cl for k in keywords):
                selected.append(c)
        return selected

    for year, prefix in [('2020', p20), ('2022', p22)]:
        ycols = [c for c in out.columns if c.startswith(prefix)]
        if ycols:
            out[f'sanlc_total_share_{year}'] = out[ycols].fillna(0.0).sum(axis=1)

        for theme, keywords in theme_keywords.items():
            tcols = cols_for_theme(ycols, keywords)
            if tcols:
                raw_share = out[tcols].fillna(0.0).sum(axis=1)
                out[f'sanlc_{theme}_share_{year}'] = raw_share
                if f'sanlc_total_share_{year}' in out.columns:
                    out[f'sanlc_{theme}_ratio_{year}'] = _safe_div(raw_share, out[f'sanlc_total_share_{year}'].fillna(0.0))

    for theme in list(theme_keywords.keys()) + ['total']:
        c20 = f'sanlc_{theme}_share_2020'
        c22 = f'sanlc_{theme}_share_2022'
        if c20 in out.columns and c22 in out.columns:
            out[f'sanlc_{theme}_delta'] = out[c22] - out[c20]

    if {'sanlc_urban_share_2020', 'sanlc_mining_share_2020', 'sanlc_cropland_share_2020'}.issubset(out.columns):
        out['human_land_share_2020'] = (
            out['sanlc_urban_share_2020'].fillna(0.0)
            + out['sanlc_mining_share_2020'].fillna(0.0)
            + out['sanlc_cropland_share_2020'].fillna(0.0)
        )

    if {'sanlc_urban_share_2022', 'sanlc_mining_share_2022', 'sanlc_cropland_share_2022'}.issubset(out.columns):
        out['human_land_share_2022'] = (
            out['sanlc_urban_share_2022'].fillna(0.0)
            + out['sanlc_mining_share_2022'].fillna(0.0)
            + out['sanlc_cropland_share_2022'].fillna(0.0)
        )

    if {'human_land_share_2020', 'human_land_share_2022'}.issubset(out.columns):
        out['human_land_share_delta'] = out['human_land_share_2022'] - out['human_land_share_2020']

    # OSM x SANLC interactions
    if {'osm_pressure_wastewater', 'sanlc_urban_share_2022'}.issubset(out.columns):
        out['ww_urban'] = out['osm_pressure_wastewater'].fillna(0.0) * out['sanlc_urban_share_2022'].fillna(0.0)

    if {'osm_pressure_farm', 'sanlc_cropland_share_2022'}.issubset(out.columns):
        out['farm_crop'] = out['osm_pressure_farm'].fillna(0.0) * out['sanlc_cropland_share_2022'].fillna(0.0)

    if {'osm_pressure_mine', 'sanlc_mining_share_2022'}.issubset(out.columns):
        out['mine_land'] = out['osm_pressure_mine'].fillna(0.0) * out['sanlc_mining_share_2022'].fillna(0.0)

    return out


_before_cols = set(df.columns)
df = engineer_features(df)
_new_cols = sorted(set(df.columns) - _before_cols)

print('Feature engineering step completed.')
print('FE_INCLUDE_CLASS_DELTAS:', FE_INCLUDE_CLASS_DELTAS)
print('Added engineered columns:', len(_new_cols))
print('Sample engineered columns:', _new_cols[:40])


Feature engineering step completed.
FE_INCLUDE_CLASS_DELTAS: False
Added engineered columns: 103
Sample engineered columns: ['aridity_idx', 'doy_cos', 'doy_sin', 'drainage_proxy', 'dry_heat', 'evap_eff', 'farm_crop', 'farm_rain', 'farm_runoff', 'human_land_share_2020', 'human_land_share_2022', 'human_land_share_delta', 'hydro_pressure_index', 'is_wet_season', 'mine_land', 'mine_runoff', 'month_cos', 'month_sin', 'osm_inv_dist_farm', 'osm_inv_dist_mine', 'osm_inv_dist_wastewater', 'osm_local_ratio_farm', 'osm_local_ratio_mine', 'osm_local_ratio_wastewater', 'osm_log_count_farm_1km', 'osm_log_count_farm_5km', 'osm_log_count_mine_1km', 'osm_log_count_mine_5km', 'osm_log_count_wastewater_1km', 'osm_log_count_wastewater_5km', 'osm_log_dist_farm', 'osm_log_dist_mine', 'osm_log_dist_wastewater', 'osm_near_farm_lt1km', 'osm_near_farm_lt5km', 'osm_near_mine_lt1km', 'osm_near_mine_lt5km', 'osm_near_wastewater_lt1km', 'osm_near_wastewater_lt5km', 'osm_pressure_farm']


C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'sanlc_{theme}_delta'] = out[c22] - out[c20]
C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'sanlc_{theme}_delta'] = out[c22] - out[c20]
C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

## Frozen feature sets

We use two frozen feature sets:

- **A** = stable base set
- **B** = base + empirical interactions

In [20]:
# Feature sets
# C: legacy benchmark 4-feature set
# FULL_NUMERIC: all numeric non-target features from aligned contract + engineered columns
BENCHMARK_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']

RESERVED_COLS = set(TARGET_COLS + ['spatial_group', 'is_pseudo_valid'])

# Base from contract (if available)
contract_base = []
if 'contract_cols' in globals() and isinstance(contract_cols, list) and len(contract_cols) > 0:
    contract_base = [c for c in contract_cols if c in df.columns and c not in RESERVED_COLS]

# IMPORTANT: include engineered columns from current dataframe as well.
# This fixes the disconnect where engineered features were not entering PRIMARY_FEATURE_SET.
data_driven_all = [c for c in df.columns if c not in RESERVED_COLS]
base_features = list(dict.fromkeys(contract_base + data_driven_all))

# Current preprocessor is numeric-only, so select numeric subset.
FULL_NUMERIC = [c for c in base_features if pd.api.types.is_numeric_dtype(df[c])]

if not FULL_NUMERIC:
    raise RuntimeError('FULL_NUMERIC feature set is empty. Check input schema/dtypes.')

FEATURE_SETS = {
    'FULL_NUMERIC': FULL_NUMERIC,
    'C': [f for f in BENCHMARK_4 if f in df.columns],
}

PRIMARY_FEATURE_SET = 'FULL_NUMERIC'


def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested


ENGINEERED_EXACT = {
    'is_wet_season',
    'aridity_idx', 'water_balance', 'evap_eff', 'runoff_ratio', 'dry_heat', 'temp_range',
    'rain_wind_event', 'soil_texture_balance', 'soil_fines', 'drainage_proxy',
    'upstream_human_pressure', 'stream_power_proxy', 'river_discharge_per_width',
    'hydro_pressure_index', 'pop_compaction_1km', 'pop_gradient', 'pop_sum_ratio_1_to_5km',
    'human_land_share_2020', 'human_land_share_2022', 'human_land_share_delta',
    'osm_total_pressure', 'farm_rain', 'farm_runoff', 'mine_runoff', 'ww_dry',
    'ww_urban', 'farm_crop', 'mine_land',
}

engineered_cols_in_full = [
    c for c in FULL_NUMERIC
    if c in ENGINEERED_EXACT
    or c.startswith(('month_', 'doy_', 'sanlc_delta_', 'sanlc_abs_delta_'))
    or c.startswith(('osm_log_count_', 'osm_log_dist_', 'osm_inv_dist_', 'osm_local_ratio_', 'osm_ring_count_', 'osm_pressure_', 'osm_near_'))
    or (c.startswith('sanlc_') and ('_share_' in c or '_ratio_' in c or c.endswith('_delta')))
]

print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features')

print('PRIMARY_FEATURE_SET:', PRIMARY_FEATURE_SET)
print('Engineered features included in FULL_NUMERIC:', len(engineered_cols_in_full))
print('Sample engineered-included columns:', engineered_cols_in_full[:40])


Feature sets ready:
FULL_NUMERIC: 289 features
C: 3 features
PRIMARY_FEATURE_SET: FULL_NUMERIC
Engineered features included in FULL_NUMERIC: 103
Sample engineered-included columns: ['month_sin', 'month_cos', 'is_wet_season', 'doy_sin', 'doy_cos', 'aridity_idx', 'water_balance', 'evap_eff', 'runoff_ratio', 'dry_heat', 'temp_range', 'rain_wind_event', 'soil_texture_balance', 'soil_fines', 'drainage_proxy', 'upstream_human_pressure', 'stream_power_proxy', 'river_discharge_per_width', 'hydro_pressure_index', 'pop_compaction_1km', 'pop_gradient', 'pop_sum_ratio_1_to_5km', 'osm_log_dist_mine', 'osm_inv_dist_mine', 'osm_near_mine_lt1km', 'osm_near_mine_lt5km', 'osm_log_count_mine_1km', 'osm_log_count_mine_5km', 'osm_local_ratio_mine', 'osm_ring_count_mine', 'osm_pressure_mine', 'osm_log_dist_farm', 'osm_inv_dist_farm', 'osm_near_farm_lt1km', 'osm_near_farm_lt5km', 'osm_log_count_farm_1km', 'osm_log_count_farm_5km', 'osm_local_ratio_farm', 'osm_ring_count_farm', 'osm_pressure_farm']


## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [21]:
def get_preprocessor(features_used):
    '''
    Build preprocessing with targeted missing-value policy:
    - SANLC percentage features -> fill missing with 0.0 (absent class)
    - other numeric features -> median imputation
    '''
    sanlc_prefixes = ('sanlc2020_pct_', 'sanlc2022_pct_')
    sanlc_cols = [c for c in features_used if c.startswith(sanlc_prefixes)]
    other_num_cols = [c for c in features_used if c not in sanlc_cols]

    transformers = []

    if other_num_cols:
        other_num_pipe = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num_other', other_num_pipe, other_num_cols))

    if sanlc_cols:
        sanlc_pipe = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num_sanlc', sanlc_pipe, sanlc_cols))

    if not transformers:
        raise RuntimeError('No features provided to preprocessor.')

    return ColumnTransformer(transformers=transformers, remainder='drop')


## Models and shortlist

This shortlist is deliberately small:
- TA: mostly stable XGB
- EC: linear empirical + one XGB challenger
- DRP: safer linear options + one shallow XGB challenger

In [22]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor


def log_wrap(model):
    # Apply log transform on targets for optional log-variant.
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )


# Anchor RF defaults
DEFAULT_RF_PARAMS = {
    'n_estimators': 600,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

# Challenger ET defaults
DEFAULT_ET_PARAMS = {
    'n_estimators': 700,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

# Challenger HGB defaults (no n_jobs in this estimator)
DEFAULT_HGB_PARAMS = {
    'max_depth': 8,
    'learning_rate': 0.05,
    'max_iter': 450,
    'min_samples_leaf': 25,
    'l2_regularization': 0.0,
    'random_state': 42,
}

MODEL_SPECS = {
    # RF anchor + regularized ladder
    'RF_n600_raw': {'kind': 'rf', 'params': {}, 'log_target': False},
    'RF_n600_Log': {'kind': 'rf', 'params': {}, 'log_target': True},
    'RF_regA_raw': {
        'kind': 'rf',
        'params': {'max_features': 0.35, 'min_samples_leaf': 10, 'min_samples_split': 20, 'max_depth': 16},
        'log_target': False,
    },
    'RF_regB_raw': {
        'kind': 'rf',
        'params': {'max_features': 0.20, 'min_samples_leaf': 20, 'min_samples_split': 50, 'max_depth': 12},
        'log_target': False,
    },
    'RF_regA_Log': {
        'kind': 'rf',
        'params': {'max_features': 0.35, 'min_samples_leaf': 10, 'min_samples_split': 20, 'max_depth': 16},
        'log_target': True,
    },

    # ET challenger + regularized
    'ET_n700_raw': {'kind': 'et', 'params': {}, 'log_target': False},
    'ET_n700_Log': {'kind': 'et', 'params': {}, 'log_target': True},
    'ET_regA_raw': {
        'kind': 'et',
        'params': {'max_features': 0.25, 'min_samples_leaf': 20, 'min_samples_split': 50, 'max_depth': 14},
        'log_target': False,
    },

    # HGB challenger + regularized ladder
    'HGB_n450_raw': {'kind': 'hgb', 'params': {}, 'log_target': False},
    'HGB_n450_Log': {'kind': 'hgb', 'params': {}, 'log_target': True},
    'HGB_regA_raw': {
        'kind': 'hgb',
        'params': {
            'learning_rate': 0.03,
            'max_leaf_nodes': 15,
            'min_samples_leaf': 80,
            'l2_regularization': 5.0,
            'early_stopping': True,
            'validation_fraction': 0.15,
            'n_iter_no_change': 30,
            'max_iter': 450,
        },
        'log_target': False,
    },
    'HGB_regB_raw': {
        'kind': 'hgb',
        'params': {
            'learning_rate': 0.05,
            'max_leaf_nodes': 31,
            'min_samples_leaf': 40,
            'l2_regularization': 1.0,
            'early_stopping': True,
            'validation_fraction': 0.15,
            'n_iter_no_change': 25,
            'max_iter': 450,
        },
        'log_target': False,
    },
}


def build_model_from_spec(spec):
    kind = spec['kind']
    params = spec.get('params', {})
    use_log = spec.get('log_target', True)

    if kind == 'rf':
        p = DEFAULT_RF_PARAMS.copy()
        p.update(params)
        base = RandomForestRegressor(**p)
    elif kind == 'et':
        p = DEFAULT_ET_PARAMS.copy()
        p.update(params)
        base = ExtraTreesRegressor(**p)
    elif kind == 'hgb':
        p = DEFAULT_HGB_PARAMS.copy()
        p.update(params)
        base = HistGradientBoostingRegressor(**p)
    else:
        raise ValueError(f'Unknown model kind: {kind}')

    return log_wrap(base) if use_log else base


MODEL_BANK = {name: build_model_from_spec(spec) for name, spec in MODEL_SPECS.items()}

# Compact target-wise sweep to keep runtime under control while adding regularization.
TARGET_MODEL_ORDER = {
    'Total Alkalinity': [
        'RF_n600_raw', 'RF_regA_raw', 'RF_regB_raw',
        'ET_n700_raw', 'ET_regA_raw',
        'HGB_n450_raw', 'HGB_regA_raw', 'HGB_regB_raw',
    ],
    'Electrical Conductance': [
        'RF_n600_raw', 'RF_regA_raw', 'RF_regB_raw',
        'ET_n700_raw', 'ET_regA_raw',
        'HGB_n450_raw', 'HGB_regA_raw', 'HGB_regB_raw',
    ],
    'Dissolved Reactive Phosphorus': [
        'RF_n600_raw', 'RF_n600_Log', 'RF_regA_raw', 'RF_regA_Log',
        'ET_n700_raw', 'ET_n700_Log',
        'HGB_n450_raw', 'HGB_n450_Log', 'HGB_regA_raw',
    ],
}

TARGET_SWEEP = {
    target: [(PRIMARY_FEATURE_SET, model_name) for model_name in TARGET_MODEL_ORDER[target]]
    for target in TARGET_COLS
}

print('Model sweep active: regularization ladder + RF baseline anchor')
display(pd.DataFrame(
    [(t, fs, m) for t, recipes in TARGET_SWEEP.items() for fs, m in recipes],
    columns=['target', 'feature_set', 'model']
))


Model sweep active: regularization ladder + RF baseline anchor


,target,feature_set,model
0,Total Alkalinity,FULL_NUMERIC,RF_n600_raw
1,Total Alkalinity,FULL_NUMERIC,RF_regA_raw
2,Total Alkalinity,FULL_NUMERIC,RF_regB_raw
3,Total Alkalinity,FULL_NUMERIC,ET_n700_raw
4,Total Alkalinity,FULL_NUMERIC,ET_regA_raw
5,Total Alkalinity,FULL_NUMERIC,HGB_n450_raw
6,Total Alkalinity,FULL_NUMERIC,HGB_regA_raw
7,Total Alkalinity,FULL_NUMERIC,HGB_regB_raw
8,Electrical Conductance,FULL_NUMERIC,RF_n600_raw
9,Electrical Conductance,FULL_NUMERIC,RF_regA_raw


## Grouped evaluation helpers

This cell:
- runs grouped out-of-fold predictions,
- reports worst-region behavior,
- saves final artifacts for finalists.

In [23]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None, show_fold_progress=False, fold_desc=None):
    # Run grouped OOF evaluation with GroupKFold on spatial clusters,
    # plus a dummy median baseline on exactly the same folds/holdout.
    d = df_local.copy()

    if 'spatial_group' not in d.columns:
        raise RuntimeError(
            'Missing `spatial_group` in df_local. Run the data-loading/split setup cell (cell 8) '
            'to create spatial groups and pseudo-holdout flags before scout/full stages.'
        )

    if allowed_regions is not None:
        allowed = set(pd.Series(allowed_regions).astype(str).tolist())
        d = d[d['spatial_group'].astype(str).isin(allowed)].copy()

    X = d[features_used].reset_index(drop=True)
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['spatial_group'].astype(str).reset_index(drop=True)

    n_groups = int(groups.nunique())
    if n_groups < 2:
        raise RuntimeError('Need at least 2 spatial groups for grouped CV.')

    n_splits = min(int(CV_N_SPLITS), n_groups)
    gkf = GroupKFold(n_splits=n_splits)

    pred = np.full(len(d), np.nan, dtype=float)
    dummy_pred = np.full(len(d), np.nan, dtype=float)

    fold_rows = []
    dummy_fold_rows = []

    fold_iterator = tqdm(
        gkf.split(X, y, groups=groups),
        total=n_splits,
        desc=(fold_desc or f'CV {target}'),
        leave=False,
        disable=not show_fold_progress,
        unit='fold'
    )

    for fold_id, (train_idx, test_idx) in enumerate(fold_iterator, start=1):
        pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])

        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        fold_pred = np.asarray(pipe.predict(X_te), dtype=float)
        pred[test_idx] = fold_pred

        fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, fold_pred))),
            'mae': float(mean_absolute_error(y_te, fold_pred)),
        })

        dummy_value = float(np.median(y_tr))
        dummy_fold_pred = np.full(len(test_idx), dummy_value, dtype=float)
        dummy_pred[test_idx] = dummy_fold_pred

        dummy_fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, dummy_fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, dummy_fold_pred))),
            'mae': float(mean_absolute_error(y_te, dummy_fold_pred)),
        })

    if np.isnan(pred).any():
        raise RuntimeError('OOF predictions contain NaN values.')
    if np.isnan(dummy_pred).any():
        raise RuntimeError('Dummy OOF predictions contain NaN values.')

    fold_df = pd.DataFrame(fold_rows)
    dummy_fold_df = pd.DataFrame(dummy_fold_rows)

    holdout_r2 = np.nan
    dummy_holdout_r2 = np.nan

    if 'is_pseudo_valid' in d.columns:
        hold_mask = d['is_pseudo_valid'].astype(bool).reset_index(drop=True)

        if hold_mask.any() and int((~hold_mask).sum()) >= 2 and int(hold_mask.sum()) >= 2:
            hold_train_groups = set(groups.loc[~hold_mask].tolist())
            hold_test_groups = set(groups.loc[hold_mask].tolist())

            if hold_train_groups.intersection(hold_test_groups):
                raise RuntimeError('Pseudo-holdout leakage detected: train and holdout share groups.')

            hold_pipe = Pipeline([
                ('preprocessor', get_preprocessor(features_used)),
                ('model', clone(estimator))
            ])

            hold_pipe.fit(X.loc[~hold_mask], y.loc[~hold_mask])
            hold_pred = np.asarray(hold_pipe.predict(X.loc[hold_mask]), dtype=float)
            holdout_r2 = float(r2_score(y.loc[hold_mask], hold_pred))

            dummy_hold_value = float(np.median(y.loc[~hold_mask]))
            dummy_hold_pred = np.full(int(hold_mask.sum()), dummy_hold_value, dtype=float)
            dummy_holdout_r2 = float(r2_score(y.loc[hold_mask], dummy_hold_pred))

    model_r2 = float(r2_score(y, pred))
    model_rmse = float(np.sqrt(mean_squared_error(y, pred)))
    model_mae = float(mean_absolute_error(y, pred))
    model_mean_fold_r2 = float(fold_df['r2'].mean()) if not fold_df.empty else np.nan
    model_min_fold_r2 = float(fold_df['r2'].min()) if not fold_df.empty else np.nan

    dummy_r2 = float(r2_score(y, dummy_pred))
    dummy_rmse = float(np.sqrt(mean_squared_error(y, dummy_pred)))
    dummy_mae = float(mean_absolute_error(y, dummy_pred))
    dummy_mean_fold_r2 = float(dummy_fold_df['r2'].mean()) if not dummy_fold_df.empty else np.nan
    dummy_min_fold_r2 = float(dummy_fold_df['r2'].min()) if not dummy_fold_df.empty else np.nan

    delta_holdout = np.nan
    if not pd.isna(holdout_r2) and not pd.isna(dummy_holdout_r2):
        delta_holdout = float(holdout_r2 - dummy_holdout_r2)

    return {
        'pred': pred,
        'rmse': model_rmse,
        'mae': model_mae,
        'r2': model_r2,
        'mean_fold_r2': model_mean_fold_r2,
        'min_fold_r2': model_min_fold_r2,
        'holdout_r2': holdout_r2,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': n_groups,
        'dummy_r2': dummy_r2,
        'dummy_rmse': dummy_rmse,
        'dummy_mae': dummy_mae,
        'dummy_mean_fold_r2': dummy_mean_fold_r2,
        'dummy_min_fold_r2': dummy_min_fold_r2,
        'dummy_holdout_r2': dummy_holdout_r2,
        'delta_r2_vs_dummy': float(model_r2 - dummy_r2),
        'delta_min_fold_r2_vs_dummy': float(model_min_fold_r2 - dummy_min_fold_r2),
        'delta_holdout_r2_vs_dummy': delta_holdout,
    }


GLOBAL_R2_WEIGHT = 0.60
HOLDOUT_R2_WEIGHT = 0.25
MIN_FOLD_R2_WEIGHT = 0.15


def compute_selection_score(overall_r2, holdout_r2, min_fold_r2=None):
    # Build a finalist selection score with holdout awareness and fold robustness.
    holdout_term = 0.0 if pd.isna(holdout_r2) else float(holdout_r2)
    min_fold_term = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)

    return float(
        GLOBAL_R2_WEIGHT * float(overall_r2) +
        HOLDOUT_R2_WEIGHT * holdout_term +
        MIN_FOLD_R2_WEIGHT * min_fold_term
    )


def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    # Fit the final full-data pipeline and save preprocessor + model artifacts.
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path



## Stage 1: Scout run

We first run only a narrow shortlist, optionally prioritizing the hardest regions.

In [24]:
SCOUT_GROUPS = None
print('Scout scope: ALL spatial groups')

rows_scout = []

for target in tqdm(TARGET_COLS, desc='Scout targets', unit='target'):
    recipes = TARGET_SWEEP[target]
    for feature_set_name, model_name in tqdm(recipes, desc=f'Scout {target}', unit='model', leave=False):
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print()
        print(f'--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_GROUPS,
                show_fold_progress=True,
                fold_desc=f'Scout CV {target_key(target)}'
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('holdout_r2', out['holdout_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('dummy_r2', out['dummy_r2'])
            mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
            mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
            if not pd.isna(out['delta_holdout_r2_vs_dummy']):
                mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
            mlflow.log_metric('cv_time_sec', dt)

            selection_score = compute_selection_score(
                overall_r2=out['r2'],
                holdout_r2=out['holdout_r2'],
                min_fold_r2=out['min_fold_r2']
            )

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'holdout_r2': out['holdout_r2'],
                'dummy_r2': out['dummy_r2'],
                'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
                'dummy_min_fold_r2': out['dummy_min_fold_r2'],
                'dummy_holdout_r2': out['dummy_holdout_r2'],
                'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
                'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
                'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
                'selection_score': selection_score,
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
            f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
            f"selection_score={selection_score:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Scout results:')
display(scout_df)



Scout scope: ALL spatial groups


Scout targets:   0%|          | 0/3 [00:00<?, ?target/s]

Scout Total Alkalinity:   0%|          | 0/8 [00:00<?, ?model/s]


--- SCOUT__RF_n600_raw__FULL_NUMERIC__TotalAlkalinity ---


Scout CV TotalAlkalinity:   0%|          | 0/5 [00:00<?, ?fold/s]

...

Scout CV DissolvedReactivePhosphorus:   0%|          | 0/5 [00:00<?, ?fold/s]

   fold     n        r2       rmse        mae
0     1  1862 -0.834377  87.185329  64.790926
1     2  1855 -0.046741  56.646660  35.382311
2     3  1844 -0.518316  55.626281  37.469446
3     4  1917 -0.034611  26.576574  14.318214
4     5  1841 -0.028413  32.515027  19.361703
OOF R2=-0.2021 vs dummy=-0.2130 (delta=0.0108) | Holdout R2=-0.0253 vs dummy=-0.0144 | selection_score=-0.2528 | min_fold_r2=-0.8344 vs dummy=-0.8584

--- SCOUT__HGB_regA_raw__FULL_NUMERIC__DissolvedReactivePhosphorus ---


Scout CV DissolvedReactivePhosphorus:   0%|          | 0/5 [00:00<?, ?fold/s]

   fold     n        r2       rmse        mae
0     1  1862 -0.576704  80.830292  63.610134
1     2  1855  0.079867  53.110453  37.477071
2     3  1844  0.135758  41.967839  30.587788
3     4  1917 -0.646077  33.522431  23.886839
4     5  1841 -0.588785  40.414091  29.162624
OOF R2=-0.0656 vs dummy=-0.2130 (delta=0.1473) | Holdout R2=0.0635 vs dummy=-0.0144 | selection_score=-0.1204 | min_fold_r2=-0.6461 vs dummy=-0.8584

Scout results:


,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,...,dummy_mean_fold_r2,dummy_min_fold_r2,dummy_holdout_r2,delta_r2_vs_dummy,delta_min_fold_r2_vs_dummy,delta_holdout_r2_vs_dummy,selection_score,rmse,mae,cv_time_sec
0,scout,SCOUT__RF_regA_raw__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,RF_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.032956,-0.141874,-0.486775,-0.024902,...,-0.275420,-0.858426,-0.014441,0.245924,0.371651,-0.010461,-0.059468,50.130416,35.821417,105.039779
1,scout,SCOUT__RF_n600_raw__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,RF_n600_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.008467,-0.175957,-0.448870,-0.031518,...,-0.275420,-0.858426,-0.014441,0.221435,0.409557,-0.017077,-0.070130,50.761196,36.787953,27.512147
2,scout,SCOUT__ET_n700_raw__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,ET_n700_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.030562,-0.192247,-0.491611,-0.100845,...,-0.275420,-0.858426,-0.014441,0.182406,0.366816,-0.086404,-0.117290,51.750586,36.593493,20.335361
3,scout,SCOUT__HGB_regA_raw__FULL_NUMERIC__DissolvedRe...,Dissolved Reactive Phosphorus,HGB_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.065637,-0.319188,-0.646077,0.063499,...,-0.275420,-0.858426,-0.014441,0.147331,0.212349,0.077940,-0.120419,52.623882,36.897240,34.751255
4,scout,SCOUT__RF_regA_Log__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,RF_regA_Log,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.093399,-0.150199,-0.737397,0.051381,...,-0.275420,-0.858426,-0.014441,0.119570,0.121030,0.065823,-0.153803,53.304935,30.983448,95.976083
5,scout,SCOUT__RF_n600_Log__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,RF_n600_Log,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.115916,-0.169830,-0.747690,0.049274,...,-0.275420,-0.858426,-0.014441,0.097053,0.110737,0.063715,-0.169384,53.851010,31.203700,26.148631
6,scout,SCOUT__ET_n700_Log__FULL_NUMERIC__DissolvedRea...,Dissolved Reactive Phosphorus,ET_n700_Log,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.138315,-0.193355,-0.758489,0.028457,...,-0.275420,-0.858426,-0.014441,0.074653,0.099937,0.042898,-0.189648,54.388794,31.280760,24.403650
7,scout,SCOUT__HGB_n450_raw__FULL_NUMERIC__DissolvedRe...,Dissolved Reactive Phosphorus,HGB_n450_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.121677,-0.454756,-0.957252,0.017310,...,-0.275420,-0.858426,-0.014441,0.091292,-0.098825,0.031751,-0.212266,53.989835,37.661625,60.742258
8,scout,SCOUT__HGB_n450_Log__FULL_NUMERIC__DissolvedRe...,Dissolved Reactive Phosphorus,HGB_n450_Log,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",-0.202125,-0.292492,-0.834377,-0.025347,...,-0.275420,-0.858426,-0.014441,0.010844,0.024049,-0.010906,-0.252768,55.892422,34.173351,56.376942
9,scout,SCOUT__RF_regA_raw__FULL_NUMERIC__ElectricalCo...,Electrical Conductance,RF_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.307156,0.219694,0.105110,0.252306,...,-0.299134,-0.756982,-0.091150,0.453612,0.862093,0.343456,0.263137,284.604211,209.801156,90.204249


## Stage 2: Full finalists

For each target, we take the top 2 scout candidates and run full grouped CV.
We also save inference artifacts for those finalists.

In [25]:
def build_finalist_shortlist(scout_df_local, top_n=2):
    # Keep a union of top candidates by complementary views so we do not
    # discard globally stronger or more robust models too early.
    pieces = []

    for target_name, tdf in scout_df_local.groupby('target'):
        by_selection = tdf.sort_values(
            ['selection_score', 'r2', 'min_fold_r2'],
            ascending=[False, False, False]
        ).head(top_n)

        by_global = tdf.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(top_n)

        by_robust = tdf.sort_values(
            ['min_fold_r2', 'r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(1)

        shortlist = pd.concat([by_selection, by_global, by_robust], axis=0)
        shortlist = shortlist.drop_duplicates(subset=['run_name']).reset_index(drop=True)
        pieces.append(shortlist)

    return pd.concat(pieces, axis=0).reset_index(drop=True)


finalist_df = build_finalist_shortlist(scout_df, top_n=2)

print('Finalists (union shortlist):')
display(finalist_df[[
    'target',
    'feature_set',
    'model_name',
    'selection_score',
    'holdout_r2',
    'r2',
    'min_fold_r2',
    'delta_r2_vs_dummy',
    'delta_holdout_r2_vs_dummy',
    'run_name'
]])

rows_full = []
OOF_PREDS = {}

for _, row in tqdm(finalist_df.iterrows(), total=len(finalist_df), desc='Full finalists', unit='run'):
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print()
    print(f'=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None,
            show_fold_progress=True,
            fold_desc=f'Full CV {target_key(target)}'
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('holdout_r2', out['holdout_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('dummy_r2', out['dummy_r2'])
        mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
        mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
        if not pd.isna(out['delta_holdout_r2_vs_dummy']):
            mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            holdout_r2=out['holdout_r2'],
            min_fold_r2=out['min_fold_r2']
        )

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'holdout_r2': out['holdout_r2'],
            'dummy_r2': out['dummy_r2'],
            'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
            'dummy_min_fold_r2': out['dummy_min_fold_r2'],
            'dummy_holdout_r2': out['dummy_holdout_r2'],
            'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
            'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
            'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
        f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
        f"selection_score={selection_score:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Full results:')
display(full_df)



Finalists (union shortlist):


,target,feature_set,model_name,selection_score,holdout_r2,r2,min_fold_r2,delta_r2_vs_dummy,delta_holdout_r2_vs_dummy,run_name
0,Dissolved Reactive Phosphorus,FULL_NUMERIC,RF_regA_raw,-0.059468,-0.024902,0.032956,-0.486775,0.245924,-0.010461,SCOUT__RF_regA_raw__FULL_NUMERIC__DissolvedRea...
1,Dissolved Reactive Phosphorus,FULL_NUMERIC,RF_n600_raw,-0.070130,-0.031518,0.008467,-0.448870,0.221435,-0.017077,SCOUT__RF_n600_raw__FULL_NUMERIC__DissolvedRea...
2,Electrical Conductance,FULL_NUMERIC,RF_regA_raw,0.263137,0.252306,0.307156,0.105110,0.453612,0.343456,SCOUT__RF_regA_raw__FULL_NUMERIC__ElectricalCo...
3,Electrical Conductance,FULL_NUMERIC,RF_regB_raw,0.246884,0.251386,0.289748,0.067925,0.436204,0.342536,SCOUT__RF_regB_raw__FULL_NUMERIC__ElectricalCo...
4,Electrical Conductance,FULL_NUMERIC,HGB_regA_raw,0.191580,0.046521,0.296314,0.014412,0.442769,0.137670,SCOUT__HGB_regA_raw__FULL_NUMERIC__ElectricalC...
5,Total Alkalinity,FULL_NUMERIC,RF_regA_raw,0.194036,0.117695,0.335253,-0.243596,0.495610,0.203419,SCOUT__RF_regA_raw__FULL_NUMERIC__TotalAlkalinity
6,Total Alkalinity,FULL_NUMERIC,HGB_regA_raw,0.189218,0.000407,0.330599,-0.061621,0.490956,0.086131,SCOUT__HGB_regA_raw__FULL_NUMERIC__TotalAlkali...


Full finalists:   0%|          | 0/7 [00:00<?, ?run/s]


=== FULL__RF_regA_raw__FULL_NUMERIC__DissolvedReactivePhosphorus ===


Full CV DissolvedReactivePhosphorus:   0%|          | 0/5 [00:00<?, ?fold/s]

...

Full CV TotalAlkalinity:   0%|          | 0/5 [00:00<?, ?fold/s]

   fold     n        r2       rmse        mae
0     1  1862  0.442294  55.381091  44.009060
1     2  1855  0.105540  72.213410  53.689907
2     3  1844  0.068278  55.473864  45.636261
3     4  1917 -0.243596  64.657845  54.417652
4     5  1841  0.107669  54.576833  37.070952
FULL OOF R2=0.3353 vs dummy=-0.1604 (delta=0.4956) | Holdout R2=0.1177 vs dummy=-0.0857 | selection_score=0.1940 | min_fold_r2=-0.2436 vs dummy=-1.3037

=== FULL__HGB_regA_raw__FULL_NUMERIC__TotalAlkalinity ===


Full CV TotalAlkalinity:   0%|          | 0/5 [00:00<?, ?fold/s]

   fold     n        r2       rmse        mae
0     1  1862  0.462087  54.389483  44.466691
1     2  1855  0.032561  75.101606  54.678411
2     3  1844 -0.008577  57.716489  45.456554
3     4  1917 -0.061621  59.740153  46.137487
4     5  1841  0.049090  56.339769  37.601829
FULL OOF R2=0.3306 vs dummy=-0.1604 (delta=0.4910) | Holdout R2=0.0004 vs dummy=-0.0857 | selection_score=0.1892 | min_fold_r2=-0.0616 vs dummy=-1.3037

Full results:


,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,...,dummy_holdout_r2,delta_r2_vs_dummy,delta_min_fold_r2_vs_dummy,delta_holdout_r2_vs_dummy,selection_score,rmse,mae,cv_time_sec,preproc_path,model_path
0,full,FULL__RF_regA_raw__FULL_NUMERIC__DissolvedReac...,Dissolved Reactive Phosphorus,RF_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.032956,-0.141874,-0.486775,-0.024902,...,-0.014441,0.245924,0.371651,-0.010461,-0.059468,50.130416,35.821417,109.403808,../models/final_deadline_mvp4\FULL__RF_regA_ra...,../models/final_deadline_mvp4\FULL__RF_regA_ra...
1,full,FULL__RF_n600_raw__FULL_NUMERIC__DissolvedReac...,Dissolved Reactive Phosphorus,RF_n600_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.008467,-0.175957,-0.448870,-0.031518,...,-0.014441,0.221435,0.409557,-0.017077,-0.070130,50.761196,36.787953,28.186643,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
2,full,FULL__RF_regA_raw__FULL_NUMERIC__ElectricalCon...,Electrical Conductance,RF_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.307156,0.219694,0.105110,0.252306,...,-0.091150,0.453612,0.862093,0.343456,0.263137,284.604211,209.801156,88.918601,../models/final_deadline_mvp4\FULL__RF_regA_ra...,../models/final_deadline_mvp4\FULL__RF_regA_ra...
3,full,FULL__RF_regB_raw__FULL_NUMERIC__ElectricalCon...,Electrical Conductance,RF_regB_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.289748,0.203532,0.067925,0.251386,...,-0.091150,0.436204,0.824908,0.342536,0.246884,288.157485,213.791433,43.435609,../models/final_deadline_mvp4\FULL__RF_regB_ra...,../models/final_deadline_mvp4\FULL__RF_regB_ra...
4,full,FULL__HGB_regA_raw__FULL_NUMERIC__ElectricalCo...,Electrical Conductance,HGB_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.296314,0.186368,0.014412,0.046521,...,-0.091150,0.442769,0.771394,0.137670,0.191580,286.822534,210.169034,36.553595,../models/final_deadline_mvp4\FULL__HGB_regA_r...,../models/final_deadline_mvp4\FULL__HGB_regA_r...
5,full,FULL__RF_regA_raw__FULL_NUMERIC__TotalAlkalinity,Total Alkalinity,RF_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.335253,0.096037,-0.243596,0.117695,...,-0.085724,0.495610,1.060133,0.203419,0.194036,60.895118,47.028563,91.516221,../models/final_deadline_mvp4\FULL__RF_regA_ra...,../models/final_deadline_mvp4\FULL__RF_regA_ra...
6,full,FULL__HGB_regA_raw__FULL_NUMERIC__TotalAlkalinity,Total Alkalinity,HGB_regA_raw,FULL_NUMERIC,"[""Latitude"", ""Longitude"", ""nir"", ""green"", ""swi...",0.330599,0.094708,-0.061621,0.000407,...,-0.085724,0.490956,1.242108,0.086131,0.189218,61.107903,45.682782,38.487311,../models/final_deadline_mvp4\FULL__HGB_regA_r...,../models/final_deadline_mvp4\FULL__HGB_regA_r...


## Freeze Manifest A

This chooses one safe anchor model per target from the full finalists.
For DRP, we prefer the safer linear models.

In [ ]:
# OPTIONAL LEGACY CONSTANTS (currently unused by active manifest logic)
# TARGET_GLOBAL_R2_FLOOR = {
#     'Total Alkalinity': 0.00,
#     'Electrical Conductance': 0.00,
#     'Dissolved Reactive Phosphorus': -0.20,
# }
#
# DRP_SAFE_MODELS = [
#     'RF_n600_raw',
#     'RF_n600_Log',
# ]


In [26]:
manifest_A = {}
manifest_B = {}


def pick_with_gate(tdf, preferred_models=None):
    # Hard gate: prefer rows that beat dummy on pseudo-holdout.
    gated = tdf[tdf['delta_holdout_r2_vs_dummy'] > 0].copy()
    pool = gated if not gated.empty else tdf.copy()

    if preferred_models:
        for model_name in preferred_models:
            pref = pool[pool['model_name'] == model_name].copy()
            if not pref.empty:
                pool = pref
                break

    pool = pool.sort_values(
        ['delta_holdout_r2_vs_dummy', 'holdout_r2', 'selection_score', 'r2'],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    if pool.empty:
        raise RuntimeError('No candidate rows available after gating/sorting.')
    return pool.iloc[0]


for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy().reset_index(drop=True)

    if tdf.empty:
        raise RuntimeError(f'No full-stage results available for target: {target}. Run Stage 2 first and verify full_df.')

    # Manifest A = safety anchor (RF-first preference)
    if target == 'Total Alkalinity':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_raw', 'RF_n600_Log'])
        chosen_B = pick_with_gate(tdf, preferred_models=None)

    elif target == 'Electrical Conductance':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_raw', 'RF_n600_Log'])
        chosen_B = pick_with_gate(tdf, preferred_models=None)

    elif target == 'Dissolved Reactive Phosphorus':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_Log', 'RF_n600_raw'])
        chosen_B = chosen_A

    manifest_A[target] = chosen_A.to_dict()
    manifest_B[target] = chosen_B.to_dict()


DRP_DELTA_HOLDOUT = float(manifest_A['Dissolved Reactive Phosphorus']['delta_holdout_r2_vs_dummy'])
DRP_USE_MODEL = DRP_DELTA_HOLDOUT > 0.0

print('DRP_USE_MODEL:', DRP_USE_MODEL, '| delta_holdout_vs_dummy:', round(DRP_DELTA_HOLDOUT, 6))

print('=== MANIFEST A (GATED, RF-ANCHORED) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_A.items()
}, indent=2))

print('\n=== MANIFEST B (GATED, BEST-AVAILABLE CHALLENGER) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_B.items()
}, indent=2))


DRP_USE_MODEL: False | delta_holdout_vs_dummy: -0.017077
=== MANIFEST A (GATED, RF-ANCHORED) ===
{
  "Total Alkalinity": {
    "run_name": "FULL__RF_regA_raw__FULL_NUMERIC__TotalAlkalinity",
    "model_name": "RF_regA_raw",
    "holdout_r2": 0.11769492844235441,
    "dummy_holdout_r2": -0.08572435561882252,
    "delta_holdout_r2_vs_dummy": 0.20341928406117693,
    "selection_score": 0.19403600348201624,
    "r2": 0.3352528296101437,
    "min_fold_r2": -0.24359617596439054
  },
  "Electrical Conductance": {
    "run_name": "FULL__RF_regA_raw__FULL_NUMERIC__ElectricalConductance",
    "model_name": "RF_regA_raw",
    "holdout_r2": 0.25230615505914045,
    "dummy_holdout_r2": -0.09114972904755358,
    "delta_holdout_r2_vs_dummy": 0.34345588410669403,
    "selection_score": 0.2631369698539579,
    "r2": 0.3071564381816837,
    "min_fold_r2": 0.10511045453441736
  },
  "Dissolved Reactive Phosphorus": {
    "run_name": "FULL__RF_n600_raw__FULL_NUMERIC__DissolvedReactivePhosphorus",
    "mod

## Diagnostic For Retraining Before Making Submission

In [ ]:
# OPTIONAL DIAGNOSTIC (disabled for faster runs)
# diag_feature_set = FEATURE_SETS.get(PRIMARY_FEATURE_SET, FEATURE_SETS.get('C', ['swir22', 'NDMI', 'MNDWI', 'pet']))
# cols = [c for c in diag_feature_set if c in df.columns] + TARGET_COLS
# display(df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).T)


## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [27]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using saved preprocessor/model artifacts.
    Raises clear errors when artifacts or features are missing.
    '''
    feats = json.loads(entry['features_used_json'])
    preproc_path = entry['preproc_path']
    model_path = entry['model_path']

    if not os.path.exists(preproc_path):
        raise FileNotFoundError(f'Missing preprocessor artifact: {preproc_path}')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Missing model artifact: {model_path}')

    missing_feats = [f for f in feats if f not in df_val_local.columns]
    if missing_feats:
        raise RuntimeError(f'Missing validation features for manifest run {entry.get("run_name", "unknown")}: {missing_feats[:20]}')

    pre = joblib.load(preproc_path)
    mdl = joblib.load(model_path)

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)


## Build Shot A / B / C

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B

In [28]:
df_val = pd.read_parquet(VALID_PATH).copy()
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

if len(df_val) != len(tpl):
    raise RuntimeError(f'Validation rows ({len(df_val)}) do not match submission template rows ({len(tpl)}).')

# Validate that all needed features exist in validation
needed_feats = set()
for t in ['Total Alkalinity', 'Electrical Conductance']:
    needed_feats.update(json.loads(manifest_A[t]['features_used_json']))
    needed_feats.update(json.loads(manifest_B[t]['features_used_json']))

if DRP_USE_MODEL:
    needed_feats.update(json.loads(manifest_A['Dissolved Reactive Phosphorus']['features_used_json']))

missing_feats = sorted([f for f in needed_feats if f not in df_val.columns])
if missing_feats:
    raise RuntimeError(f'Missing validation features: {missing_feats}')


def clip_by_train_quantile(pred, target, q_hi=0.995):
    hi = float(df[target].quantile(q_hi))
    return np.clip(np.asarray(pred, dtype=float), 0, hi)


drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())

# -------------------------
# Shot A: safe anchor
# -------------------------
shotA = tpl.copy()

pred_ta_a = predict_from_manifest_entry(manifest_A['Total Alkalinity'], df_val)
pred_ec_a = predict_from_manifest_entry(manifest_A['Electrical Conductance'], df_val)

shotA['Total Alkalinity'] = clip_by_train_quantile(pred_ta_a, 'Total Alkalinity')
shotA['Electrical Conductance'] = clip_by_train_quantile(pred_ec_a, 'Electrical Conductance')

if DRP_USE_MODEL:
    drp_a = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    shotA['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_a, 'Dissolved Reactive Phosphorus')
    print('Shot A DRP model:', manifest_A['Dissolved Reactive Phosphorus']['run_name'])
else:
    shotA['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot A DRP fallback: train median')

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# -------------------------
# Shot B: modest challenger
# -------------------------
shotB = tpl.copy()
shotB['Total Alkalinity'] = shotA['Total Alkalinity']

if manifest_B['Electrical Conductance']['run_name'] != manifest_A['Electrical Conductance']['run_name']:
    ec_safe = shotA['Electrical Conductance'].values
    ec_chal = predict_from_manifest_entry(manifest_B['Electrical Conductance'], df_val)
    ec_blend = 0.35 * ec_safe + 0.65 * ec_chal
    shotB['Electrical Conductance'] = clip_by_train_quantile(ec_blend, 'Electrical Conductance')
    print('Shot B EC challenger blend:', manifest_B['Electrical Conductance']['run_name'])
else:
    shotB['Electrical Conductance'] = shotA['Electrical Conductance']
    print('Shot B EC kept from Manifest A')

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend = 0.20 * drp_model + 0.80 * drp_train_median
    shotB['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend, 'Dissolved Reactive Phosphorus')
    print('Shot B DRP model+median alpha=0.20')
else:
    shotB['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot B DRP fallback: train median')

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# -------------------------
# Shot C: stronger DRP hedge
# -------------------------
shotC = tpl.copy()
shotC['Total Alkalinity'] = clip_by_train_quantile(
    0.85 * shotA['Total Alkalinity'] + 0.15 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotC['Electrical Conductance'] = clip_by_train_quantile(
    0.50 * shotA['Electrical Conductance'] + 0.50 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend_c = 0.10 * drp_model + 0.90 * drp_train_median
    shotC['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend_c, 'Dissolved Reactive Phosphorus')
    print('Shot C DRP model+median alpha=0.10')
else:
    shotC['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot C DRP fallback: train median')

assert_submission_integrity(shotC, tpl, TARGET_COLS)

# -------------------------
# Save
# -------------------------
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_challenger_blend.csv'
pathC = f'../data/submission/submission_{stamp}_C_hedge.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)



C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'sanlc_{theme}_delta'] = out[c22] - out[c20]
C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f'sanlc_{theme}_delta'] = out[c22] - out[c20]
C:\Users\USER\AppData\Local\Temp\ipykernel_10292\2395241272.py:292: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

Shot A DRP fallback: train median
Shot B EC kept from Manifest A
Shot B DRP fallback: train median
Shot C DRP fallback: train median
Saved files:
A: ../data/submission/submission_20260313_0309_A_safe.csv
B: ../data/submission/submission_20260313_0309_B_challenger_blend.csv
C: ../data/submission/submission_20260313_0309_C_hedge.csv


## Submission diagnostics

This final cell prints simple distribution summaries for the three submission variants.

In [ ]:
# OPTIONAL SUBMISSION DIAGNOSTICS (disabled for faster runs)
# def summarize_shot(shot_df, name):
#     print(f'\n{name} stats')
#     stats = shot_df[TARGET_COLS].describe(
#         percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
#     ).T
#     display(stats[['min', '1%', '5%', '50%', 'mean', '95%', '99%', 'max']])
#
# summarize_shot(shotA, 'Shot A')
# summarize_shot(shotB, 'Shot B')
# summarize_shot(shotC, 'Shot C')
#
# print('\nMean absolute deltas vs Shot A')
# delta_tbl = pd.DataFrame({
#     'target': TARGET_COLS,
#     'B_vs_A_mae': [float(np.mean(np.abs(shotB[t] - shotA[t]))) for t in TARGET_COLS],
#     'C_vs_A_mae': [float(np.mean(np.abs(shotC[t] - shotA[t]))) for t in TARGET_COLS],
# })
# display(delta_tbl)
#
# tracker = pd.DataFrame([
#     {'file': pathA, 'hypothesis': 'Safety anchor (lowest variance)'},
#     {'file': pathC, 'hypothesis': 'Balanced hedge between A and B'},
#     {'file': pathB, 'hypothesis': 'Most aggressive on EC/DRP challenger blend'},
# ])
#
# print('\nSubmission tracker:')
# display(tracker)
#
# print('\nSuggested upload order: A -> C -> B')
